In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
import keyring
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import geoplot as gplt
import plotly.express as px

In [ ]:
# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [ ]:
communities = gpd.read_file("shp_society_thrive_msp2040_com_des")
communities = communities.to_crs("EPSG:4326")
communities.plot()

In [ ]:
df = pd.read_csv(data_dir + "/data_processed/feasible_shifts.csv")
df

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["home_lon"], df["home_lat"]), crs="EPSG:4326")
gdf

In [ ]:
gdf["community"] = "na"
for i, row in communities.iterrows():
    gdf["community"] = np.where(gdf["geometry"].within(row["geometry"]), row.name, gdf["community"])
gdf["community"].value_counts()

In [ ]:
valid_modes = [("Car","distance"), 
               ("Bike/Scooter","distance"), 
               ("Walk","distance"), 
               ("Transit","distance")]

def plot_mode_density(df: pd.DataFrame, modes=valid_modes, percentile=0.95, size=(12, 6), bins=300, function=lambda x: x, ax=None, fig=None):
    palette = itertools.cycle(sns.color_palette()) # cycle through colors to make sure each mode gets a unique one
    if (ax == None):
        fig, ax = plt.subplots(figsize=size)
    for m in modes: # cycle through all modes
        mode = m[0]
        column = m[1]
        label = mode + ' ' + column
        
        c = next(palette) # get color to use
        group = df[df["mode"] == mode] # filter out the current mode
        sns.histplot(function(group[column]), ax=ax, stat="density", kde=True, label=label, color=c, bins=bins) # plot hist plot with kde overlayed in the color
        val = function(group[column]).quantile(q=percentile) # calculate the value of the given percentile (default 0.95)
        ax.axvline(x=val, color=c) # plot line representing that value on the plot        
        ax.legend()
    return fig, ax

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
rg=LinearSegmentedColormap.from_list('rg',["r", "w", "g"], N=256) 
rg.set_bad(color="grey")

In [ ]:
gdf_subset = gdf.loc[gdf["community"] != "na", :].copy()

In [ ]:
communities["geometry"] = (
        communities.to_crs(communities.estimate_utm_crs()).simplify(50).to_crs(communities.crs)
    )

In [ ]:
communities

In [ ]:
def view(percentile):
    # fig, ax = plot_mode_density(df, [('Walk','walk_distance_miles'), ('Car','walk_distance_miles')], percentile=percentile)
    # ax.set_xlim(left=0, right=40)
    
    #| output: true

    # set the maximum feasible walking distance to WALK_DISTANCE_CUTOFF (default 1.6 miles)
    gdf_subset["within_feasible_walking_dist"] = True
    
    cutoff = gdf[gdf["mode"] == "Walk"]["walk_distance_miles"].quantile(percentile)

    gdf_subset.loc[gdf_subset['walk_distance_miles'] > cutoff, 'feasible_walk_shift'] = False
    gdf_subset.loc[gdf_subset["walk_distance_miles"] > cutoff, "within_feasible_walking_dist"] = False
    
    communities["val"] = (gdf_subset.groupby("community")["within_feasible_walking_dist"].sum() / gdf_subset.groupby("community")["within_feasible_walking_dist"].count()).fillna(0)
    
    # gplt.choropleth(communities, hue=communities["val"], projection=gplt.crs.AlbersEqualArea(), cmap=rg, figsize=(12, 12))
    return px.choropleth(communities, geojson=communities.geometry, locations=communities.index, color="val")

In [ ]:
x = view(0.95)
x.update_geos(fitbounds="locations")
x.show()

In [ ]:
type(x)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
#| output: true
%matplotlib inline
from ipywidgets import interact, FloatSlider

interact(view, percentile=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.95))